## **Prompt Chaining LangGraph WorkFlow**

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [11]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",temperature=0.0, max_tokens=3000)

In [7]:
# Define a TypedDict for the state data
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    eval: float

In [8]:
# Define a function to create an outline for the blog
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state



# Define a function to write the blog content
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

# Define a function to evaluate the blog content
def evaluate_blog(state:BlogState) -> BlogState:

    content = state['content']

    prompt = f'Evaluate the following blog content for the topic - {state["title"]} on a scale of 1 to 10 with step of 0.5 and provide only the score. \n {content}'

    eval_score = model.invoke(prompt).content

    state["eval"] = float(eval_score)

    return state

In [9]:
#Define the graph
graph = StateGraph(BlogState)

# add nodes

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog',evaluate_blog)

#add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

# compile workflow

workflow = graph.compile()


In [12]:
#define initial state

initial_state = {'title' : "UPSC Vs RBI Grade B Comprehensive Comparision Summary"}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'UPSC Vs RBI Grade B Comprehensive Comparision Summary', 'outline': '## Blog Outline: UPSC Vs. RBI Grade B: A Comprehensive Comparison Summary\n\n**Blog Title Options:**\n\n* UPSC vs. RBI Grade B: Which is the Right Path for You? A Detailed Comparison\n* Cracking the Code: UPSC Civil Services vs. RBI Grade B Officer - A Comprehensive Guide\n* The Ultimate Showdown: UPSC vs. RBI Grade B - A Comparative Analysis for Aspiring Officers\n* Beyond the Hype: A Realistic Comparison of UPSC and RBI Grade B Officer Roles\n\n**Target Audience:** Aspirants preparing for either UPSC Civil Services Exam or RBI Grade B Officer Exam, students exploring career options, and individuals interested in public sector careers.\n\n**Blog Goal:** To provide a clear, concise, and comprehensive comparison of the UPSC Civil Services Exam and the RBI Grade B Officer Exam, helping aspirants make informed decisions about their career path.\n\n---\n\n**I. Introduction (Approx. 200-300 words)**\n\n*   **Hook

In [13]:
final_state['content']

"## UPSC vs. RBI Grade B: Which is the Right Path for You? A Detailed Comparison\n\nThe post-graduation dilemma for ambitious Indian graduates often boils down to a crucial choice: the prestigious administrative ladder of the UPSC Civil Services Exam (CSE) or the impactful role within India's central banking institution, the Reserve Bank of India (RBI) Grade B Officer Exam. Both paths promise a career of national significance, immense responsibility, and substantial growth. However, they diverge significantly in their nature, examination process, and the day-to-day realities of the roles. This blog aims to demystify these two formidable examinations, offering a comprehensive, side-by-side comparison to empower you in making an informed decision about your future. We'll delve into everything from eligibility and exam patterns to the roles, responsibilities, and career trajectories, helping you chart the course that best aligns with your aspirations.\n\n### Understanding the Core: What a

In [14]:
final_state['eval']

8.5

In [15]:
final_state['outline']

'## Blog Outline: UPSC Vs. RBI Grade B: A Comprehensive Comparison Summary\n\n**Blog Title Options:**\n\n* UPSC vs. RBI Grade B: Which is the Right Path for You? A Detailed Comparison\n* Cracking the Code: UPSC Civil Services vs. RBI Grade B Officer - A Comprehensive Guide\n* The Ultimate Showdown: UPSC vs. RBI Grade B - A Comparative Analysis for Aspiring Officers\n* Beyond the Hype: A Realistic Comparison of UPSC and RBI Grade B Officer Roles\n\n**Target Audience:** Aspirants preparing for either UPSC Civil Services Exam or RBI Grade B Officer Exam, students exploring career options, and individuals interested in public sector careers.\n\n**Blog Goal:** To provide a clear, concise, and comprehensive comparison of the UPSC Civil Services Exam and the RBI Grade B Officer Exam, helping aspirants make informed decisions about their career path.\n\n---\n\n**I. Introduction (Approx. 200-300 words)**\n\n*   **Hook:** Start with a relatable scenario – the dilemma faced by many ambitious grad